In [ ]:
%run "./00_CC_Mapping_Setup.ipynb"

In [ ]:
%sql
/* ===================================================================================
   METRIC: EBA07 - Marketing (Canada, Hong Kong, Barbados Only)
   
   BUSINESS QUESTION:
   Did the AU incur marketing expenses during the assessment period?
   
   LOGIC SUMMARY:
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
      Only AUs in this filtered list (Canada, HK, Barbados + US_OR_CANADA = 'CANADA') 
      will appear in the final report.
   2. FINANCE DATA ONLY: Unions the 6 Finance files. Coupa data is excluded.
   3. CATEGORY CODE FILTER: Strictly filters the Finance data for Marketing codes 
      ('194', '370', '372', '377', '408', '628', '629', '653', '830', '875', '877').
   4. MAPPING AGGREGATION: Joins the Cost Center Mapping Bootstrap to the transactions 
      to determine which AUs had marketing spend. Evaluates ALL mapped AUs.
   5. FINAL OUTPUT: LEFT JOINS the mapping results onto the Master List anchor and
      returns marketing transaction count.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data: This is our strict anchor for the final output
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Combined_Finance_Raw AS (
    -- 2. Union all 6 Finance files into one master dataset
    SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_1
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_2
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_3
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_4
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_5
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_6
),

All_Finance_Transactions AS (
    -- 3. Clean all finance rows
    SELECT 
        CASE WHEN TRIM(CostCenter) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(CostCenter), 4), 4, '0') ELSE UPPER(TRIM(CostCenter)) END AS Cleaned_CC,
        TRIM(CatCode) AS CatCode
    FROM Combined_Finance_Raw
    WHERE CostCenter IS NOT NULL
),

Marketing_Transactions AS (
    -- 4. Filter STRICTLY for marketing category codes
    SELECT Cleaned_CC, CatCode
    FROM All_Finance_Transactions
    WHERE TRIM(CatCode) IN (
        '194', '370', '372', '377', '408', '628', 
        '629', '653', '830', '875', '877'
    )
),

CC_Mapping AS (
    -- Standardize the CC Mapping table using the correct cost_center_id column
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        CASE WHEN TRIM(cost_center_id) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(cost_center_id), 4), 4, '0') ELSE UPPER(TRIM(cost_center_id)) END AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
),

AU_Marketing_Flags AS (
    -- 5a. Count marketing findings per AU
    SELECT 
        m.AU_ID,
        COUNT(mt.Cleaned_CC) AS Marketing_Row_Count
    FROM CC_Mapping m
    LEFT JOIN Marketing_Transactions mt
        ON m.Mapped_CC = mt.Cleaned_CC
    GROUP BY m.AU_ID
)

-- 5. Build final report using the Filtered Master List as the anchor
SELECT 
    a.BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'EBA07' AS QuestionID,
    COALESCE(amf.Marketing_Row_Count, 0) AS Response,
    a.In_ABAC_AU_List
FROM Master_AUs a
LEFT JOIN AU_Marketing_Flags amf
    ON a.BusinessID = amf.AU_ID
ORDER BY a.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: EBA07 - AU Level Calculation Review
   PURPOSE: One row per AU showing normalized Cost Centers and how the final count
   response was calculated.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Combined_Finance_Raw AS (
    SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_1
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_2
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_3
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_4
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_5
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_6
),

All_Finance_Transactions AS (
    SELECT 
        CASE WHEN TRIM(CostCenter) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(CostCenter), 4), 4, '0') ELSE UPPER(TRIM(CostCenter)) END AS Cleaned_CC,
        TRIM(CatCode) AS CatCode
    FROM Combined_Finance_Raw
    WHERE CostCenter IS NOT NULL
),

Marketing_Transactions AS (
    SELECT Cleaned_CC, CatCode
    FROM All_Finance_Transactions
    WHERE TRIM(CatCode) IN ('194', '370', '372', '377', '408', '628', '629', '653', '830', '875', '877')
),

CC_Mapping AS (
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        CASE WHEN TRIM(cost_center_id) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(cost_center_id), 4), 4, '0') ELSE UPPER(TRIM(cost_center_id)) END AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
),

Mapped_CC_By_AU AS (
    SELECT 
        AU_ID,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(Mapped_CC))) AS Normalized_CC_List,
        COUNT(*) AS Mapped_CC_Count
    FROM CC_Mapping
    GROUP BY AU_ID
),

Mapped_All_Finance AS (
    SELECT 
        m.AU_ID,
        f.Cleaned_CC,
        f.CatCode
    FROM CC_Mapping m
    INNER JOIN All_Finance_Transactions f
        ON m.Mapped_CC = f.Cleaned_CC
),

Mapped_Marketing AS (
    SELECT 
        m.AU_ID,
        mt.Cleaned_CC,
        mt.CatCode
    FROM CC_Mapping m
    INNER JOIN Marketing_Transactions mt
        ON m.Mapped_CC = mt.Cleaned_CC
),

Marketing_By_AU AS (
    SELECT 
        AU_ID,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(Cleaned_CC))) AS Marketing_CC_List,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(CatCode))) AS Marketing_CatCode_List,
        COUNT(*) AS Marketing_Row_Count
    FROM Mapped_Marketing
    GROUP BY AU_ID
),

Finance_Record_By_AU AS (
    SELECT 
        AU_ID,
        COUNT(*) AS Mapped_Row_Count
    FROM Mapped_All_Finance
    GROUP BY AU_ID
)

SELECT 
    mast.BusinessID,
    mast.AU_Name,
    mast.Business_Segment,
    'EBA07' AS QuestionID,
    COALESCE(mark.Marketing_Row_Count, 0) AS Response,
    COALESCE(map.Normalized_CC_List, 'n/a') AS Normalized_CC_List,
    COALESCE(mark.Marketing_CC_List, 'n/a') AS Marketing_CC_List,
    COALESCE(mark.Marketing_CatCode_List, 'n/a') AS Marketing_CatCode_List,
    COALESCE(map.Mapped_CC_Count, 0) AS Mapped_CC_Count,
    COALESCE(rec.Mapped_Row_Count, 0) AS Mapped_Row_Count,
    COALESCE(mark.Marketing_Row_Count, 0) AS Marketing_Row_Count,
    COALESCE(mark.Marketing_Row_Count, 0) AS Final_Count,
    CONCAT('Response = ', CAST(COALESCE(mark.Marketing_Row_Count, 0) AS STRING), ' marketing rows with target category codes. Total mapped finance rows=', CAST(COALESCE(rec.Mapped_Row_Count, 0) AS STRING), '.') AS Calculation_Formula,
    mast.In_ABAC_AU_List
FROM Master_AUs mast
LEFT JOIN Mapped_CC_By_AU map
    ON mast.BusinessID = map.AU_ID
LEFT JOIN Marketing_By_AU mark
    ON mast.BusinessID = mark.AU_ID
LEFT JOIN Finance_Record_By_AU rec
    ON mast.BusinessID = rec.AU_ID
ORDER BY mast.BusinessID;


In [ ]:
%sql
/* ===================================================================================
   QA DIFF LOGIC: Missing AUs for Canada, Barbados, Hong Kong (EBA07)
   
   PURPOSE: 
   Identifies Business IDs that exist in abac_2025.fy25_list_of_unit for specific 
   jurisdictions but have NO matching marketing transactions in the EBA07 logic.
=================================================================================== */

WITH Filtered_Master_List AS (
    -- 1. Grab the Master AUs and filter jurisdiction for the target countries
    SELECT 
        TRIM(CAST(business_ID AS STRING)) AS BusinessID,
        TRIM(business_operational_unit_name) AS AU_Name,
        TRIM(jurisdiction) AS jurisdiction
    FROM abac_2025.fy25_list_of_unit
    WHERE business_ID IS NOT NULL
      AND TRIM(jurisdiction) IN ('Canada', 'Hong Kong', 'Barbados')
),

Combined_Finance_Raw AS (
    -- 2. Pull the 6 Finance files based on your EBA07 logic
    SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_1
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_2
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_3
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_4
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_5
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_6
),

Marketing_Transactions AS (
    -- 3. Filter for Marketing Category Codes
    SELECT 
        CASE WHEN TRIM(CostCenter) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(CostCenter), 4), 4, '0') ELSE UPPER(TRIM(CostCenter)) END AS Cleaned_CC
    FROM Combined_Finance_Raw
    WHERE TRIM(CatCode) IN (
        '194', '370', '372', '377', '408', '628', 
        '629', '653', '830', '875', '877'
    )
),

EBA07_Successful_AUs AS (
    -- 4. Get the distinct Business IDs that successfully matched to a marketing transaction
    SELECT DISTINCT 
        TRIM(CAST(m.AU_ID AS STRING)) AS BusinessID
    FROM vw_cost_center_mapping_bootstrap m
    INNER JOIN Marketing_Transactions mt
        ON CASE WHEN TRIM(m.Cost_Center_ID) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(m.Cost_Center_ID), 4), 4, '0') ELSE UPPER(TRIM(m.Cost_Center_ID)) END = mt.Cleaned_CC
)

-- 5. Find the Difference (What is in the Filtered Master List but NOT in EBA07 Successful AUs)
SELECT 
    fml.BusinessID AS Missing_BusinessID,
    fml.AU_Name,
    fml.jurisdiction,
    'No EBA07 Marketing Transactions Mapped' AS Diff_Reason
FROM Filtered_Master_List fml
LEFT JOIN EBA07_Successful_AUs eba 
    ON fml.BusinessID = eba.BusinessID
WHERE eba.BusinessID IS NULL
ORDER BY fml.BusinessID;